# Seção 3.5 — Testes de estresse da escada de modelos

**Por que esta seção existe.** KNN e Random Forest chegaram a 100% na validação por blocos. Quando dois modelos batem no teto, a comparação entre eles não informa nada. Para aprender algo, precisamos de testes que *possam falhar*.

**Regras:** usamos só o TREINO (`X_train`, `meta_train`, `cv_folds`). O `X_test_locked` **não é tocado** aqui.

**Os 5 testes:**
1. Recall da anomalia e falso alarme **por velocidade** (a métrica primária do plano, ainda não reportada).
2. **Sonda da frequência 1X:** a rotação sozinha separa normal de fita? (detecta confusão de sessão)
3. **Divisão cronológica:** treina no começo da gravação e valida no fim.
4. **Janelas aleatórias × blocos**, com os rótulos reais.
5. **Rótulos embaralhados por bloco**, repetido várias vezes.

**Pré-requisitos já na memória do notebook:** `X_train`, `meta_train`, `y_bin_train`, `y_6_train`, `grupos_train`, `cv_folds`, `modelos`, `resultados_predicoes` e, para o Teste 2, os dados brutos (`df_raw_normal`, `df_raw_fita`, `cortes_normal`, `cortes_fita`).

**Cuidado ao interpretar:** cada validação tem poucos blocos. Trate diferenças pequenas como ruído.

## Teste 1 — Recall da anomalia e falso alarme por velocidade

**O que faz:** usa as predições fora do fold (`resultados_predicoes`) que você já tem e calcula, para cada modelo, o recall da anomalia e a taxa de falso alarme (FPR) separados por velocidade.

**Por quê:** o relatório global esconde onde o modelo falha. A hipótese a verificar é que os modelos simples falham na Vel 1 com fita (a energia dela fica dentro da faixa dos normais). Isso é uma pergunta, não uma conclusão.

**Como ler:**
- *Recall* = das janelas com fita, que fração foi detectada.
- *FPR* = das janelas normais, que fração foi marcada como anomalia (queremos baixo).
- Recall alto na Vel 2 e 3 e baixo na Vel 1 indica que o modelo só enxerga energia absoluta.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

vel_train = meta_train['velocidade'].values


def metricas_por_velocidade(y_real, y_pred, vel):
    """Recall da anomalia e FPR por velocidade, mais o total."""
    saida = {}
    for v in [1, 2, 3, 'Total']:
        m = np.ones(len(y_real), dtype=bool) if v == 'Total' else (vel == v)
        anom, norm = (y_real == 1) & m, (y_real == 0) & m
        rot = 'Total' if v == 'Total' else f'Vel {v}'
        saida[f'Recall {rot}'] = ((y_pred == 1) & anom).sum() / anom.sum()
        saida[f'FPR {rot}'] = ((y_pred == 1) & norm).sum() / norm.sum()
    return saida


def tabela_metricas(predicoes, y_real, vel):
    return pd.DataFrame({nome: metricas_por_velocidade(y_real, p, vel) for nome, p in predicoes.items()}).T


def heat_recall(tab, titulo):
    cols = ['Recall Vel 1', 'Recall Vel 2', 'Recall Vel 3', 'Recall Total']
    plt.figure(figsize=(8.5, 0.55 * len(tab) + 1.8))
    sns.heatmap(tab[cols] * 100, annot=True, fmt='.0f', cmap='Blues', vmin=0, vmax=100, cbar=False)
    plt.title(titulo, fontweight='bold')
    plt.tight_layout()
    plt.show()


tab_bloco = tabela_metricas(resultados_predicoes, y_bin_train, vel_train)
display(tab_bloco.round(3))
heat_recall(tab_bloco, "Recall da anomalia por velocidade (%) — validação cruzada por blocos")

## Teste 2 — Sonda da frequência de rotação (1X)

**O que faz:** para cada janela, estima a frequência de rotação (o maior pico entre 8 e 30 Hz, com zero-padding para afinar a resolução) e mostra como ela evolui no tempo. Depois treina um classificador de **uma única regra** (um *stump*) usando *só* essa frequência, dentro de cada velocidade, com os mesmos folds por blocos.

**Por quê:** os dois ensaios podem ter rotações diferentes por motivos que não são a fita (o ensaio com fita começou com o motor já aquecido, e o normal aqueceu durante a gravação). Se a rotação sozinha separar normal de fita, o modelo pode estar usando a sessão, e não o defeito.

**Como ler:**
- *Gráfico:* se as duas cores ocupam faixas de frequência diferentes numa velocidade, os ensaios diferem nela.
- *Tabela:* acerto próximo de 50% = a rotação não distingue as classes. Acerto alto = há confusão de sessão naquela velocidade. Isso não invalida o resultado, mas precisa constar como limitação e justifica novas coletas.

`f1x_hz` é só metadado de diagnóstico e **não entra no modelo**.

In [ ]:
from sklearn.tree import DecisionTreeClassifier

N_JAN, FATOR_ZP, FS = 5000, 8, 1000                       # janela de 5 s; zero-padding 8x -> ~0,025 Hz
hann = np.hanning(N_JAN)
freqs = np.fft.rfftfreq(N_JAN * FATOR_ZP, d=1 / FS)
faixa_1x = (freqs >= 8) & (freqs <= 30)

arr_normal = df_raw_normal[['x', 'y', 'z']].values        # convertidos uma única vez (mais rápido)
arr_fita = df_raw_fita[['x', 'y', 'z']].values


def freq_1x(sinal_xyz):
    """Frequência do maior pico entre 8 e 30 Hz, somando os espectros dos 3 eixos."""
    esp = 0
    for k in range(3):
        s = sinal_xyz[:, k] - sinal_xyz[:, k].mean()
        esp = esp + np.abs(np.fft.rfft(s * hann, n=N_JAN * FATOR_ZP))
    return freqs[faixa_1x][np.argmax(esp[faixa_1x])]


def f1x_das_janelas(meta):
    f = np.zeros(len(meta))
    for i, (v, a, idx) in enumerate(zip(meta['velocidade'], meta['anomalia'], meta['idx_janela_classe'])):
        arr, cortes = (arr_fita, cortes_fita) if a == 1 else (arr_normal, cortes_normal)
        ini = cortes[v][0] * FS + idx * N_JAN              # mesma posição usada na extração
        f[i] = freq_1x(arr[ini: ini + N_JAN])
    return f


meta_train['f1x_hz'] = f1x_das_janelas(meta_train)

# --- gráfico: rotação ao longo do tempo, por velocidade ---
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, v in zip(axes, [1, 2, 3]):
    for a, cor, nome in [(0, '#1f77b4', 'Normal'), (1, '#d95f02', 'Fita')]:
        s = meta_train[(meta_train['velocidade'] == v) & (meta_train['anomalia'] == a)]
        ax.scatter(s['tempo_relativo_s'], s['f1x_hz'], s=14, color=cor, label=nome, alpha=.8)
    ax.set_title(f"Vel {v}: frequência de rotação (1X) por janela", fontweight='bold', fontsize=11)
    ax.set_xlabel("Tempo dentro da gravação (s)"); ax.set_ylabel("Hz")
    ax.grid(True, linestyle='--', alpha=.4); ax.legend()
plt.tight_layout(); plt.show()


# --- sonda: stump usando só f1x, com os folds por blocos, dentro de cada velocidade ---
def sonda_1x(v):
    m = (meta_train['velocidade'] == v).values
    f = meta_train['f1x_hz'].values.reshape(-1, 1)
    pred = np.full(len(y_bin_train), -1)
    for tr, va in cv_folds:
        tr, va = tr[m[tr]], va[m[va]]
        stump = DecisionTreeClassifier(max_depth=1, random_state=42).fit(f[tr], y_bin_train[tr])
        pred[va] = stump.predict(f[va])
    y, p = y_bin_train[m], pred[m]
    return {'Velocidade': v,
            'Acerto só com f1x': (y == p).mean(),
            'Recall anomalia': ((p == 1) & (y == 1)).sum() / (y == 1).sum(),
            'FPR': ((p == 1) & (y == 0)).sum() / (y == 0).sum(),
            'Janelas': int(m.sum())}


display(pd.DataFrame([sonda_1x(v) for v in [1, 2, 3]]).round(3))

## Teste 3 — Divisão cronológica (treina no começo, valida no fim)

**O que faz:** treina cada modelo da escada nos blocos iniciais de cada gravação (blocos 0 a 6) e valida nos blocos finais (8 e 9). O bloco 7 pertence ao teste trancado e fica **fora dos dois lados**, funcionando como um fosso natural no tempo.

**Por quê:** na validação cruzada por blocos, os blocos de validação têm vizinhos de treino de ambos os lados no tempo, e a deriva térmica pode ajudar o modelo. Aqui o modelo precisa *extrapolar* para a parte final da gravação (motor mais quente).

**Como ler:** compare com o Teste 1.
- Queda pequena = o modelo não depende tanto da deriva.
- Queda grande = parte do desempenho vinha de vizinhança temporal ou do estado térmico.
- A queda também pode ter causas que não são a fita (o motor realmente mudou), por isso este é um **teste de estresse, não uma nota**.
- A validação tem só 2 blocos por classe, então os números são grosseiros.

In [ ]:
from sklearn.base import clone

tr_cron = (meta_train['bloco_local'] <= 6).values
va_cron = (meta_train['bloco_local'] >= 8).values
print(f"Treino cronológico: {tr_cron.sum()} janelas | Validação final: {va_cron.sum()} janelas")

pred_cron = {}
for nome, (pipe, X_dados) in modelos.items():
    p = clone(pipe).fit(X_dados[tr_cron], y_bin_train[tr_cron])
    pred_cron[nome] = p.predict(X_dados[va_cron])

tab_cron = tabela_metricas(pred_cron, y_bin_train[va_cron], vel_train[va_cron])
display(tab_cron.round(3))
heat_recall(tab_cron, "Recall da anomalia por velocidade (%) — divisão cronológica (início → fim)")

## Teste 4 — Janelas aleatórias × blocos (rótulos reais)

**O que faz:** repete a validação de cada modelo de duas formas e compara o acerto: (a) por **blocos** (o que já fizemos) e (b) por **janelas sorteadas** (o jeito ingênuo, estratificado pelas 6 classes).

**Por quê:** mostra, nos *seus* dados, quanto a vizinhança temporal infla a nota.

**Como ler:**
- Modelos fracos tendem a mostrar a diferença. Se a versão ingênua der bem mais que a por blocos, o vazamento é real e justifica o cuidado.
- Modelos já no teto (100%) não mostram diferença, e isso **não** significa que o vazamento não existe: apenas que não há margem para aparecer. O Teste 5 é mais sensível para isso.

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_predict

cv_janela = list(StratifiedKFold(n_splits=4, shuffle=True, random_state=42).split(X_train, y_6_train))

linhas = []
for nome, (pipe, X_dados) in modelos.items():
    p_janela = cross_val_predict(pipe, X_dados, y_bin_train, cv=cv_janela)
    linhas.append({'Modelo': nome,
                   'Acerto por blocos': (resultados_predicoes[nome] == y_bin_train).mean(),
                   'Acerto por janelas (ingênuo)': (p_janela == y_bin_train).mean()})
tab_vaza = pd.DataFrame(linhas).set_index('Modelo')
display((tab_vaza * 100).round(1))

ax = tab_vaza.plot(kind='barh', figsize=(9, 0.7 * len(tab_vaza) + 1.8), color=['#1f77b4', '#d95f02'])
ax.set_xlim(0, 1.05); ax.set_xlabel('Acerto global')
ax.set_title("Mesma validação, dois jeitos de dividir", fontweight='bold')
ax.grid(True, axis='x', linestyle='--', alpha=.4)
plt.tight_layout(); plt.show()

## Teste 5 — Rótulos embaralhados por bloco

**O que faz:** troca os rótulos entre os 48 blocos de treino (cada bloco inteiro recebe o rótulo de outro bloco, sorteado) e roda a validação de novo, repetindo várias vezes. Faz isso com os dois jeitos de dividir: por **blocos** e por **janelas**.

**Por quê:** com rótulos sem relação com os sinais, um pipeline correto deve cair para o acaso (~50% no binário). Já uma divisão por janelas continua "acertando", porque a vizinha do mesmo bloco, que está no treino, carrega o mesmo rótulo falso. Isso mede o vazamento temporal.

**Como ler:**
- Histograma azul (por blocos) centrado perto de 50% = pipeline sem vazamento temporal.
- Histograma laranja (por janelas) muito acima de 50% = o vazamento que o plano evita.
- A linha tracejada é o acerto com os rótulos **reais** por blocos. Se ela fica bem à direita da distribuição azul, o sinal é real dentro da sessão.
- Este teste **não detecta** confusão de sessão (a classe é a própria gravação). Para isso servem o Teste 2 e uma sessão nova.
- Com poucas repetições, o menor p-valor possível é 1/(N+1).

In [ ]:
N_REP = 50                                                   # aumente para 100 se o tempo permitir
escolhidos = ['N1: Regressão Logística (Tempo)', 'N2: KNN k=5 (99 Features)', 'N2: Random Forest (99 Features)']

blocos = pd.DataFrame({'bloco': grupos_train, 'y': y_bin_train}).groupby('bloco')['y'].first()   # 48 blocos
rng = np.random.default_rng(42)
nulo = {n: {'bloco': [], 'janela': []} for n in escolhidos}

for rep in range(N_REP):
    mapa = dict(zip(blocos.index, rng.permutation(blocos.values)))       # embaralha ENTRE blocos
    y_perm = pd.Series(grupos_train).map(mapa).values
    cv_naive = list(StratifiedKFold(n_splits=4, shuffle=True, random_state=rep).split(X_train, y_6_train))
    for nome in escolhidos:
        pipe, X_dados = modelos[nome]
        p_b = cross_val_predict(pipe, X_dados, y_perm, cv=cv_folds)
        p_j = cross_val_predict(pipe, X_dados, y_perm, cv=cv_naive)
        nulo[nome]['bloco'].append((p_b == y_perm).mean())
        nulo[nome]['janela'].append((p_j == y_perm).mean())

fig, axes = plt.subplots(1, len(escolhidos), figsize=(5.4 * len(escolhidos), 4), sharey=True)
for ax, nome in zip(axes, escolhidos):
    real = (resultados_predicoes[nome] == y_bin_train).mean()
    nb = np.array(nulo[nome]['bloco'])
    p_valor = (np.sum(nb >= real) + 1) / (N_REP + 1)
    ax.hist(nulo[nome]['bloco'], bins=15, range=(0, 1), alpha=.85, color='#1f77b4', label='rótulos embaralhados, por blocos')
    ax.hist(nulo[nome]['janela'], bins=15, range=(0, 1), alpha=.6, color='#d95f02', label='rótulos embaralhados, por janelas')
    ax.axvline(real, color='k', ls='--', label='rótulos reais, por blocos')
    ax.set_title(f"{nome}\np-valor (blocos) = {p_valor:.3f}", fontsize=10, fontweight='bold')
    ax.set_xlabel('Acerto global'); ax.set_xlim(0, 1.02)
axes[0].set_ylabel('Nº de repetições'); axes[0].legend(fontsize=8, loc='upper left')
plt.tight_layout(); plt.show()

resumo = pd.DataFrame({n: {'média por blocos': np.mean(v['bloco']), 'desvio por blocos': np.std(v['bloco']),
                           'média por janelas': np.mean(v['janela'])} for n, v in nulo.items()}).T
display((resumo * 100).round(1))

## Como ler os cinco testes juntos

| Resultado | O que indica | O que fazer |
|---|---|---|
| Teste 1: modelos simples caem na Vel 1, KNN/RF não | A informação para a Vel 1 existe, mas exige combinar features (FFT ou interação com a velocidade) | Reforça a cascata ou o uso da velocidade como entrada |
| Teste 2: acerto ~50% | A rotação não distingue as classes | Confusão de sessão por rotação é improvável nessa velocidade |
| Teste 2: acerto alto | Os ensaios diferem na rotação | Registrar como limitação e priorizar novas coletas |
| Teste 3: queda grande | Parte do desempenho vem de vizinhança ou deriva térmica | Investigar por classe e comparar com a sessão nova |
| Teste 4/5: por janelas muito acima de por blocos | Vazamento temporal real; a divisão por blocos era necessária | Citar como justificativa metodológica |
| Teste 5: azul perto de 50% e linha real bem à direita | Sem vazamento temporal; sinal real dentro da sessão | Seguir, lembrando que a generalização entre sessões continua sem teste |

**Se KNN e RF seguirem no teto em todos os testes**, a Seção 4 (hiperparâmetros) tem pouco a ensinar. Prefira usar o tempo em: modelos só com as 9 features de tempo (para ver se a FFT é necessária), a tarefa de 6 classes e, principalmente, uma sessão nova de coleta como teste externo.